# Tutorial 09 · Build an Encoder–Decoder Transformer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_09_Transformers_Implementation/tutorial_09_pytorch.ipynb)

Implement a small Spanish → English Transformer, inspect attention and masks, train with teacher forcing, and translate autoregressively.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Tutorials 03 and 08, tensor shapes, embeddings, and gradient descent.

**Resources:** CPU-compatible quick mode; a 2.6 MB archive; 3,000/400/400 sentence pairs and six epochs. Larger mode is optional. Allow 60–90 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals and scope

- Trace token IDs → embeddings + positions → encoder → masked decoder → vocabulary logits.
- Distinguish encoder self-attention, causal decoder self-attention, and cross-attention.
- Separate PAD, UNK, BOS, and EOS; shift targets for teacher forcing; ignore PAD in metrics.
- Select a checkpoint with validation loss, then evaluate held-out teacher-forced predictions **and** free-running generation.
- Inspect attention without treating it as an explanation of causality; save and reload a complete inference checkpoint.

**Correction to the source lesson:** the upstream notebook titled “Transformers Implementation” actually uses GRUs plus attention. This adaptation keeps its translation task but implements a genuine attention-based Transformer without recurrence. It is a small teaching model, not a production translator or a pretrained language model. A quick run can produce poor translations even when every implementation check passes.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import math, re, json, time, random, platform, hashlib, unicodedata, urllib.request, zipfile
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from IPython.display import display
%matplotlib inline
SEED=42
QUICK_RUN=True
DEVICE_PREFERENCE="auto"  # "cpu" forces CPU.
assert DEVICE_PREFERENCE in {"auto","cpu"}
device=torch.device("cuda" if DEVICE_PREFERENCE=="auto" and torch.cuda.is_available() else "cpu")
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
PAD,UNK,BOS,EOS=0,1,2,3
SPECIALS=["<pad>","<unk>","<bos>","<eos>"]
MAX_WORDS=12;VOCAB_LIMIT=3000;BATCH_SIZE=64
EPOCHS=6 if QUICK_RUN else 15
DATA_DIR=Path("data/tutorial_09");OUTPUT_DIR=Path("outputs/tutorial_09")
DATA_DIR.mkdir(parents=True,exist_ok=True);OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
versions={"python":platform.python_version(),"torch":str(torch.__version__),"numpy":np.__version__,
          "pandas":pd.__version__,"matplotlib":matplotlib.__version__,"device":str(device),"seed":SEED,"quick_run":QUICK_RUN}
display(pd.Series(versions,name="Environment"))
run_started=time.perf_counter()

## 2. Download, normalize, and split once

We use the same Google-hosted snapshot of the ManyThings/Tatoeba English–Spanish sentence pairs as the source lesson, over HTTPS. Hosting under a TensorFlow download URL does **not** require TensorFlow. The archive includes dataset attribution; retain it when reusing the data.

NFC normalization preserves Spanish accents and ñ. Lowercasing and a simple word/punctuation tokenizer still lose information. Keep sentences with 2–12 tokens. To prevent the same normalized sentence appearing in different partitions, shuffle deterministically and retain at most one pair per unique Spanish sentence **and** unique English sentence. This deliberately discards alternative translations; it does not remove every near-duplicate. Split this retained pool once; fit both vocabularies only on training pairs. This is an instructional split, not a standard translation benchmark.

In [ ]:
DATA_URL="https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
archive_path=DATA_DIR/"spa-eng.zip"
if not archive_path.exists():
    partial=archive_path.with_suffix(".part")
    with urllib.request.urlopen(DATA_URL,timeout=120) as response:partial.write_bytes(response.read())
    partial.replace(archive_path)
archive_sha256=hashlib.sha256(archive_path.read_bytes()).hexdigest()
with zipfile.ZipFile(archive_path) as archive:
    member=next(name for name in archive.namelist() if name.endswith("/spa.txt") or name=="spa.txt")
    raw=archive.read(member).decode("utf-8")
    for name in archive.namelist():
        if "_about" in name.lower() or "readme" in name.lower():
            (DATA_DIR/Path(name).name).write_bytes(archive.read(name))
TOKEN_PATTERN=r"[^\W\d_]+(?:'[^\W\d_]+)?|[.?!,¿¡]"
def tokenize(text):return re.findall(TOKEN_PATTERN,unicodedata.normalize("NFC",text).lower())
assert "está" in tokenize("¿Todavía está en casa?") and "niño" in tokenize("El niño.")
candidates=[]
for line in raw.splitlines():
    fields=line.split("\t")
    if len(fields)<2:continue
    english,spanish=map(tokenize,fields[:2])
    if 2<=len(english)<=MAX_WORDS and 2<=len(spanish)<=MAX_WORDS:
        candidates.append((spanish,english))
rng=random.Random(SEED);rng.shuffle(candidates)
seen_src=set();seen_tgt=set();unique=[]
for src,tgt in candidates:
    if tuple(src) in seen_src or tuple(tgt) in seen_tgt:continue
    seen_src.add(tuple(src));seen_tgt.add(tuple(tgt));unique.append((src,tgt))
sizes=(3000,400,400) if QUICK_RUN else (12000,1500,1500)
assert len(unique)>=sum(sizes)
a,b,c=sizes
train_pairs=unique[:a];val_pairs=unique[a:a+b];test_pairs=unique[a+b:a+b+c]
for side in (0,1):
    sets=[{tuple(pair[side]) for pair in part} for part in (train_pairs,val_pairs,test_pairs)]
    assert all(sets[i].isdisjoint(sets[j]) for i in range(3) for j in range(i+1,3))
def build_vocab(pairs,side):
    counts=Counter(token for pair in pairs for token in pair[side])
    words=[w for w,_ in sorted(counts.items(),key=lambda kv:(-kv[1],kv[0]))[:VOCAB_LIMIT-4]]
    return {word:i for i,word in enumerate(SPECIALS+words)}
src_vocab=build_vocab(train_pairs,0);tgt_vocab=build_vocab(train_pairs,1)
src_words=list(src_vocab);tgt_words=list(tgt_vocab)
assert [src_vocab[w] for w in SPECIALS]==[PAD,UNK,BOS,EOS]
assert [tgt_vocab[w] for w in SPECIALS]==[PAD,UNK,BOS,EOS]
print("Filtered / unique pairs:",len(candidates),len(unique))
print("Train / validation / test:",sizes,"Vocabulary sizes:",len(src_vocab),len(tgt_vocab))
print("Archive SHA-256:",archive_sha256)
display(pd.DataFrame([{"Spanish":" ".join(s),"English":" ".join(t)} for s,t in train_pairs[:3]]))

## 3. IDs, padding, and shifted targets

Each sentence is encoded as `BOS words EOS`. Teacher forcing feeds `BOS words` into the decoder and predicts `words EOS`. A causal mask still matters: without it, a decoder could read later target tokens and cheat. PAD=0 is never UNK=1. Padded target positions contribute neither loss nor accuracy.

The heatmaps below show arbitrary IDs and which source positions are blocked as attention keys. A padding mask removes **key columns**, not necessarily query rows; we ignore padded output positions in the loss.

In [ ]:
def encode(tokens,vocab):return torch.tensor([BOS]+[vocab.get(w,UNK) for w in tokens]+[EOS],dtype=torch.long)
def collate(batch):
    sources=[encode(src,src_vocab) for src,tgt in batch]
    targets=[encode(tgt,tgt_vocab) for src,tgt in batch]
    return (pad_sequence(sources,batch_first=True,padding_value=PAD),
            pad_sequence([t[:-1] for t in targets],batch_first=True,padding_value=PAD),
            pad_sequence([t[1:] for t in targets],batch_first=True,padding_value=PAD))
shuffle_generator=torch.Generator().manual_seed(SEED)
def make_loader(pairs,shuffle=False):
    return DataLoader(pairs,batch_size=BATCH_SIZE,shuffle=shuffle,collate_fn=collate,num_workers=0,
                      generator=shuffle_generator if shuffle else None)
train_loader=make_loader(train_pairs,True);val_loader=make_loader(val_pairs)
example_src,example_in,example_out=collate(train_pairs[:3])
for row,(_,target) in enumerate(train_pairs[:3]):
    length=len(target)+1
    assert example_in[row,0]==BOS and example_out[row,length-1]==EOS
    assert torch.equal(example_in[row,1:length],example_out[row,:length-1])
fig,axes=plt.subplots(1,2,figsize=(10,3),layout="constrained")
axes[0].imshow(example_src,cmap="viridis",aspect="auto");axes[0].set(title="Source token IDs (arbitrary numbers)",xlabel="Position",ylabel="Example")
axes[1].imshow(example_src.eq(PAD),cmap="gray_r",vmin=0,vmax=1,aspect="auto")
axes[1].set(title="Blocked source keys: black = PAD",xlabel="Position",ylabel="Example")
fig.savefig(OUTPUT_DIR/"token_padding.png",dpi=140);plt.show()
display(pd.DataFrame({"decoder input":[tgt_words[i] for i in example_in[0].tolist()],
                      "next-token target":[tgt_words[i] for i in example_out[0].tolist()]}))

## 4. Scaled dot-product attention and causal masking

For one head, `scores = Q @ K.T / sqrt(d_k)`, then `weights = softmax(scores)` and `output = weights @ V`. Each query gets a distribution over allowed keys. Multiple heads use separate learned projections and concatenate their outputs.

Here **True means blocked** for our masks and for `nn.MultiheadAttention`. In the functional `scaled_dot_product_attention` boolean API, True means **allowed**; the equivalence check explicitly inverts the mask. The diagonal remains allowed: position t may see itself and previous positions, but not future inputs.

In [ ]:
def causal_mask(length,device=None):return torch.ones(length,length,dtype=torch.bool,device=device).triu(1)
q=torch.randn(1,1,5,8);k=torch.randn_like(q);v=torch.randn_like(q)
blocked=causal_mask(5)
scores=(q@k.transpose(-2,-1))/math.sqrt(q.shape[-1])
weights=scores.masked_fill(blocked,float("-inf")).softmax(-1)
manual=weights@v
reference=F.scaled_dot_product_attention(q,k,v,attn_mask=~blocked,dropout_p=0.0)
torch.testing.assert_close(manual,reference)
torch.testing.assert_close(weights.sum(-1),torch.ones_like(weights.sum(-1)))
assert weights[0,0][blocked].abs().max()==0
fig,axes=plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
axes[0].imshow(blocked,cmap="gray_r",vmin=0,vmax=1);axes[0].set(title="Causal mask: black = blocked",xlabel="Key position",ylabel="Query position")
p=axes[1].imshow(weights[0,0],vmin=0,vmax=1,cmap="Blues");axes[1].set(title="Attention over allowed keys",xlabel="Key position",ylabel="Query position")
fig.colorbar(p,ax=axes[1]);fig.savefig(OUTPUT_DIR/"causal_attention.png",dpi=140);plt.show()
print("Manual attention matches PyTorch; masked weights are zero and rows sum to one.")

## 5. Add position information

Self-attention alone does not encode token order. Add fixed sinusoidal positions to scaled learned token embeddings. Dimension 2i uses `sin(position / 10000^(2i/d))`; dimension 2i+1 uses cosine. Position encoding is not a word embedding and is not trained here.

In [ ]:
class PositionEncoding(nn.Module):
    def __init__(self,dimension,max_length=64):
        super().__init__();assert dimension%2==0
        position=torch.arange(max_length).float()[:,None]
        frequency=torch.exp(torch.arange(0,dimension,2).float()*(-math.log(10000.0)/dimension))
        table=torch.zeros(max_length,dimension)
        table[:,0::2]=torch.sin(position*frequency);table[:,1::2]=torch.cos(position*frequency)
        self.register_buffer("table",table)
    def forward(self,x):
        assert x.size(1)<=len(self.table)
        return x+self.table[:x.size(1)].to(x.dtype)
position_demo=PositionEncoding(32)
assert torch.equal(position_demo.table[0,0::2],torch.zeros(16))
assert torch.equal(position_demo.table[0,1::2],torch.ones(16))
assert not torch.equal(position_demo.table[0],position_demo.table[1])
fig,ax=plt.subplots(figsize=(9,3),layout="constrained")
p=ax.imshow(position_demo.table[:24],aspect="auto",cmap="coolwarm",vmin=-1,vmax=1)
ax.set(xlabel="Embedding dimension",ylabel="Position",title="Sinusoidal position encodings")
fig.colorbar(p,ax=ax);fig.savefig(OUTPUT_DIR/"positions.png",dpi=140);plt.show()

## 6. Assemble the Transformer

We implement residual connections, layer normalization, feed-forward blocks, and masks explicitly, using PyTorch's multi-head attention primitive. This is one encoder layer and one decoder layer by default. It has no RNN. The decoder's three sublayers are masked self-attention, cross-attention over encoder outputs, and a position-wise feed-forward network.

| Block | Queries | Keys and values | Mask |
|---|---|---|---|
| Encoder self-attention | Spanish states | Spanish states | Source PAD |
| Decoder self-attention | English prefix | English prefix | Future positions + target PAD |
| Decoder cross-attention | Decoder states | Encoded Spanish | Source PAD |

Post-norm residual blocks and dropout follow the basic architecture, but this small experiment omits the original paper's large model, learning-rate schedule, label smoothing, and beam search.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self,d,heads,ff,dropout):
        super().__init__()
        self.attention=nn.MultiheadAttention(d,heads,dropout=dropout,batch_first=True)
        self.ff=nn.Sequential(nn.Linear(d,ff),nn.ReLU(),nn.Dropout(dropout),nn.Linear(ff,d))
        self.norm1=nn.LayerNorm(d);self.norm2=nn.LayerNorm(d);self.drop=nn.Dropout(dropout)
    def forward(self,x,padding,inspect=False):
        y,weights=self.attention(x,x,x,key_padding_mask=padding,need_weights=inspect,average_attn_weights=False)
        x=self.norm1(x+self.drop(y));x=self.norm2(x+self.drop(self.ff(x)))
        return x,weights
class DecoderBlock(nn.Module):
    def __init__(self,d,heads,ff,dropout):
        super().__init__()
        self.self_attention=nn.MultiheadAttention(d,heads,dropout=dropout,batch_first=True)
        self.cross_attention=nn.MultiheadAttention(d,heads,dropout=dropout,batch_first=True)
        self.ff=nn.Sequential(nn.Linear(d,ff),nn.ReLU(),nn.Dropout(dropout),nn.Linear(ff,d))
        self.norm1=nn.LayerNorm(d);self.norm2=nn.LayerNorm(d);self.norm3=nn.LayerNorm(d);self.drop=nn.Dropout(dropout)
    def forward(self,x,memory,target_padding,source_padding,inspect=False):
        y,self_weights=self.self_attention(x,x,x,attn_mask=causal_mask(x.size(1),x.device),
            key_padding_mask=target_padding,need_weights=inspect,average_attn_weights=False)
        x=self.norm1(x+self.drop(y))
        y,cross_weights=self.cross_attention(x,memory,memory,key_padding_mask=source_padding,
            need_weights=inspect,average_attn_weights=False)
        x=self.norm2(x+self.drop(y));x=self.norm3(x+self.drop(self.ff(x)))
        return x,self_weights,cross_weights
class TransformerTranslator(nn.Module):
    def __init__(self,source_size,target_size,d=64,heads=4,ff=128,dropout=0.1):
        super().__init__();self.d=d
        self.source_embedding=nn.Embedding(source_size,d,padding_idx=PAD)
        self.target_embedding=nn.Embedding(target_size,d,padding_idx=PAD)
        self.position=PositionEncoding(d);self.drop=nn.Dropout(dropout)
        self.encoder=EncoderBlock(d,heads,ff,dropout);self.decoder=DecoderBlock(d,heads,ff,dropout)
        self.output=nn.Linear(d,target_size)
    def forward(self,source,target,inspect=False):
        source_padding=source.eq(PAD);target_padding=target.eq(PAD)
        memory,encoder_weights=self.encoder(self.drop(self.position(self.source_embedding(source)*math.sqrt(self.d))),source_padding,inspect)
        decoded,self_weights,cross_weights=self.decoder(self.drop(self.position(self.target_embedding(target)*math.sqrt(self.d))),
            memory,target_padding,source_padding,inspect)
        return self.output(decoded),{"encoder":encoder_weights,"decoder":self_weights,"cross":cross_weights}
config={"source_size":len(src_vocab),"target_size":len(tgt_vocab),"d":64,"heads":4,"ff":128,"dropout":0.1}
torch.manual_seed(SEED);model=TransformerTranslator(**config).to(device)
print("Trainable parameters:",sum(p.numel() for p in model.parameters()))

## 7. Check behavior before training

Shape checks are necessary but insufficient. Change future decoder tokens and verify earlier logits stay fixed. Append PAD tokens and verify valid logits stay fixed. Inspect actual attention weights in evaluation mode, where dropout is disabled. These tests catch information leakage and padding bugs that a decreasing training loss could hide.

In [ ]:
model.eval()
s=example_src.to(device);t=example_in.to(device)
with torch.no_grad():
    logits,maps=model(s,t,inspect=True)
    assert logits.shape==(*t.shape,len(tgt_vocab)) and torch.isfinite(logits).all()
    changed=t.clone();changed[:,2:]=UNK
    changed_logits,_=model(s,changed)
    torch.testing.assert_close(logits[:,:2],changed_logits[:,:2],rtol=1e-4,atol=1e-5)
    padded_logits,_=model(F.pad(s,(0,3)),F.pad(t,(0,2)))
    torch.testing.assert_close(logits,padded_logits[:,:t.size(1)],rtol=1e-4,atol=1e-5)
    for name,attention in maps.items():
        torch.testing.assert_close(attention.sum(-1),torch.ones_like(attention.sum(-1)),rtol=1e-5,atol=1e-6)
    assert maps["decoder"].masked_select(causal_mask(t.size(1),device)[None,None]).abs().max()<1e-7
    assert maps["cross"].masked_select(s.eq(PAD)[:,None,None,:]).abs().max()<1e-7
print("Causal invariance, padding invariance, attention normalization, and blocked attention checks passed.")

## 8. Train with token-weighted metrics

Cross-entropy is summed over non-PAD target tokens, then divided by their count. Epoch metrics aggregate those sums and counts, so short final batches do not get the same weight as full batches. Teacher-forced accuracy sees the correct prefix at each position; it is **not translation accuracy**. EOS and UNK targets are included; we report lexical-token accuracy and OOV rates separately later.

Adam, gradient clipping, and a small fixed epoch budget keep this manageable. Restore the lowest-validation-loss checkpoint before any test evaluation. Do not tune settings after reading test results.

In [ ]:
def token_statistics(logits,labels):
    loss_sum=F.cross_entropy(logits.reshape(-1,logits.size(-1)),labels.reshape(-1),ignore_index=PAD,reduction="sum")
    mask=labels.ne(PAD);correct=((logits.argmax(-1)==labels)&mask).sum()
    return loss_sum,correct,mask.sum()
def epoch_pass(loader,optimizer=None):
    model.train(optimizer is not None);loss_total=correct_total=count=0
    with torch.set_grad_enabled(optimizer is not None):
        for source,target_in,target_out in loader:
            source,target_in,target_out=[x.to(device) for x in (source,target_in,target_out)]
            logits,_=model(source,target_in);loss_sum,correct,number=token_statistics(logits,target_out)
            loss=loss_sum/number;assert torch.isfinite(loss)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True);loss.backward()
                norm=nn.utils.clip_grad_norm_(model.parameters(),1.0);assert torch.isfinite(norm)
                optimizer.step()
            loss_total+=loss_sum.item();correct_total+=correct.item();count+=number.item()
    return {"loss":loss_total/count,"token_accuracy":correct_total/count,"tokens":count}
initial_validation=epoch_pass(val_loader)
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
history=[];best_loss=float("inf");best_epoch=0
for epoch in range(1,EPOCHS+1):
    training=epoch_pass(train_loader,optimizer);validation=epoch_pass(val_loader)
    history.append({"epoch":epoch,"train_loss":training["loss"],"val_loss":validation["loss"],
                    "train_accuracy":training["token_accuracy"],"val_accuracy":validation["token_accuracy"]})
    if validation["loss"]<best_loss:
        best_loss=validation["loss"];best_epoch=epoch
        best_state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
    print(f"Epoch {epoch}: train loss {training['loss']:.3f}, validation loss {validation['loss']:.3f}, teacher-forced accuracy {validation['token_accuracy']:.1%}")
model.load_state_dict(best_state);model.eval()
history=pd.DataFrame(history)
fig,axes=plt.subplots(1,2,figsize=(10,3.5),layout="constrained")
for prefix,label in [("train","Training"),("val","Validation")]:
    axes[0].plot(history.epoch,history[prefix+"_loss"],label=label)
    axes[1].plot(history.epoch,history[prefix+"_accuracy"],label=label)
for ax in axes:ax.axvline(best_epoch,color="gray",ls="--");ax.set_xlabel("Epoch");ax.legend()
axes[0].set_ylabel("Cross-entropy per non-PAD token");axes[1].set(ylabel="Teacher-forced token accuracy",ylim=(0,1))
fig.savefig(OUTPUT_DIR/"training.png",dpi=140);plt.show()
print("Restored validation-selected epoch:",best_epoch)

## 9. Greedy decoding: use your own previous predictions

Start with BOS and repeatedly predict the next token. EOS stops a sentence; completed rows receive PAD while other rows continue. We disallow generating BOS or PAD for unfinished rows. A length cap prevents infinite generation, and we record whether EOS was reached. This simple implementation recomputes the encoder/prefix each step; caching and beam search are outside scope.

The tiny scripted check below tests termination independently of learned quality. Poor models must still handle EOS correctly.

In [ ]:
def advance(next_ids,done):
    emitted=torch.where(done,torch.full_like(next_ids,PAD),next_ids)
    return emitted,done|emitted.eq(EOS)
x,done=advance(torch.tensor([EOS,7]),torch.tensor([False,False]))
y,done=advance(torch.tensor([8,EOS]),done)
assert x.tolist()==[EOS,7] and y.tolist()==[PAD,EOS] and done.all()
@torch.no_grad()
def greedy(network,source,max_steps=MAX_WORDS+1):
    network.eval();prefix=torch.full((len(source),1),BOS,device=source.device,dtype=torch.long)
    done=torch.zeros(len(source),dtype=torch.bool,device=source.device)
    for _ in range(max_steps):
        logits,_=network(source,prefix);next_logits=logits[:,-1].clone();next_logits[:,[PAD,BOS]]=float("-inf")
        emitted,done=advance(next_logits.argmax(-1),done);prefix=torch.cat([prefix,emitted[:,None]],dim=1)
        if done.all():break
    return prefix[:,1:],done
def decoded_words(ids,words):
    result=[]
    for idx in ids:
        if idx==EOS:break
        if idx not in (PAD,BOS):result.append(words[idx])
    return result

In [ ]:
test_loader=make_loader(test_pairs)
test_teacher_forced=epoch_pass(test_loader)
# Prespecified constant-token baseline, fitted only on training targets.
training_target_ids=[idx for _,target in train_pairs for idx in encode(target,tgt_vocab).tolist()[1:]]
majority_id=Counter(training_target_ids).most_common(1)[0][0]
baseline_correct=total=known_correct=known_total=0
rows=[];eos_count=exact_count=0
with torch.no_grad():
    offset=0
    for source,target_in,target_out in test_loader:
        source,target_in,target_out=[x.to(device) for x in (source,target_in,target_out)]
        logits,_=model(source,target_in);mask=target_out.ne(PAD);lexical=target_out.ge(4)
        baseline_correct+=((target_out==majority_id)&mask).sum().item();total+=mask.sum().item()
        known_correct+=((logits.argmax(-1)==target_out)&lexical).sum().item();known_total+=lexical.sum().item()
        generated,ended=greedy(model,source);eos_count+=ended.sum().item()
        for j,ids in enumerate(generated.cpu().tolist()):
            src,reference=test_pairs[offset+j];prediction=decoded_words(ids,tgt_words)
            # Compare actual normalized words, not UNK-collapsed reference IDs.
            exact=prediction==reference and bool(ended[j]);exact_count+=int(exact)
            rows.append({"Spanish":" ".join(src),"reference":" ".join(reference),"generated":" ".join(prediction),"reached_EOS":bool(ended[j]),"exact_match":exact})
        offset+=len(source)
def oov_rate(pairs,side,vocab):
    words=[w for pair in pairs for w in pair[side]]
    return sum(w not in vocab for w in words)/len(words)
evaluation={"teacher_forced":test_teacher_forced,"constant_token_baseline_accuracy":baseline_correct/total,
    "baseline_token":tgt_words[majority_id],"known_lexical_token_accuracy":known_correct/known_total,
    "greedy_exact_match":exact_count/len(test_pairs),"greedy_eos_rate":eos_count/len(test_pairs),
    "test_source_oov":oov_rate(test_pairs,0,src_vocab),"test_target_oov":oov_rate(test_pairs,1,tgt_vocab)}
print(json.dumps(evaluation,indent=2));display(pd.DataFrame(rows[:8]))
# Predictions are runtime artifacts; no dataset sentences are committed to the repository.
pd.DataFrame(rows).to_csv(OUTPUT_DIR/"test_predictions.csv",index=False)
assert len(pd.read_csv(OUTPUT_DIR/"test_predictions.csv"))==len(test_pairs)

Read teacher-forced and free-running results separately. Exact match is strict: one valid alternative wording counts as wrong with our single reference. Low OOV coverage and short training further limit interpretation. Inspect the first eight fixed examples above rather than selecting the best-looking predictions. This tutorial does not claim a BLEU benchmark or useful translation quality.

## 10. Inspect trained attention

The following maps use the **first validation pair** and its reference prefix (teacher forcing), not generated output. We show one decoder self-attention head and all four cross-attention heads. Cross-attention rows are decoder input positions; each row contributes to predicting the **next** token. A bright cell means high attention weight in that head, not a verified word alignment or causal explanation.

In [ ]:
vis_src,vis_in,vis_out=collate(val_pairs[:1])
with torch.no_grad():_,attention=model(vis_src.to(device),vis_in.to(device),inspect=True)
s_labels=[src_words[i] for i in vis_src[0].tolist()];t_labels=[tgt_words[i] for i in vis_in[0].tolist()]
fig,axes=plt.subplots(1,5,figsize=(20,4.5),layout="constrained")
matrices=[attention["decoder"][0,0].cpu()]+[attention["cross"][0,h].cpu() for h in range(config["heads"])]
for i,(ax,matrix) in enumerate(zip(axes,matrices)):
    ax.imshow(matrix,vmin=0,vmax=1,cmap="Blues",aspect="auto")
    labels=t_labels if i==0 else s_labels
    ax.set(xticks=range(len(labels)),xticklabels=labels,yticks=range(len(t_labels)),yticklabels=t_labels,
           title="Decoder self-attention, head 1" if i==0 else f"Cross-attention, head {i}")
    ax.tick_params(axis="x",rotation=90,labelsize=8);ax.tick_params(axis="y",labelsize=8)
    ax.set_xlabel("Key token");ax.set_ylabel("Decoder input query")
fig.suptitle("Teacher-forced validation example; common color scale 0–1")
fig.savefig(OUTPUT_DIR/"trained_attention.png",dpi=140);plt.show()

## 11. Save, reload on CPU, and verify

Weights alone cannot translate text. Save the source/target vocabularies, special IDs, tokenizer settings, maximum length, model configuration, and selected epoch too. Reconstruct the same model on CPU and compare logits and greedy output. Load only checkpoints you trust.

In [ ]:
checkpoint={"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},"config":config,
            "source_vocab":src_vocab,"target_vocab":tgt_vocab,"special_ids":{"PAD":PAD,"UNK":UNK,"BOS":BOS,"EOS":EOS},
            "token_pattern":TOKEN_PATTERN,"normalization":"NFC + lowercase","max_words":MAX_WORDS,"best_epoch":best_epoch}
torch.save(checkpoint,OUTPUT_DIR/"transformer_translation.pt")
saved=torch.load(OUTPUT_DIR/"transformer_translation.pt",map_location="cpu",weights_only=True)
restored=TransformerTranslator(**saved["config"]).eval();restored.load_state_dict(saved["state_dict"])
assert saved["source_vocab"]==src_vocab and saved["target_vocab"]==tgt_vocab
with torch.no_grad():
    original,_=model(example_src.to(device),example_in.to(device));reloaded,_=restored(example_src,example_in)
    torch.testing.assert_close(original.cpu(),reloaded,rtol=1e-4,atol=1e-5)
    a,a_done=greedy(model,example_src.to(device));b,b_done=greedy(restored,example_src)
    assert torch.equal(a.cpu(),b) and torch.equal(a_done.cpu(),b_done)
    padded,_=restored(F.pad(example_src,(0,3)),F.pad(example_in,(0,2)))
    torch.testing.assert_close(reloaded,padded[:,:example_in.size(1)],rtol=1e-4,atol=1e-5)
    loss,_,_=token_statistics(reloaded,example_out)
    padded_loss,_,_=token_statistics(padded,F.pad(example_out,(0,2)))
    torch.testing.assert_close(loss,padded_loss,rtol=1e-4,atol=1e-5)
report={"environment":versions,"data_url":DATA_URL,"archive_sha256":archive_sha256,
    "filtered_pairs":len(candidates),"unique_pairs":len(unique),"split_sizes":list(sizes),"config":config,
    "parameters":sum(p.numel() for p in model.parameters()),"epochs":EPOCHS,"best_epoch":best_epoch,
    "initial_validation":initial_validation,"best_validation_loss":best_loss,"test":evaluation,
    "checks":{"unicode":True,"split_separation_both_languages":True,"target_shift":True,"manual_attention":True,
              "causal_invariance":True,"padding_invariance":True,"masked_attention":True,"finite_training":True,
              "eos_termination":True,"cpu_checkpoint_logits":True,"cpu_checkpoint_generation":True,"masked_loss":True},
    "seconds_after_imports":time.perf_counter()-run_started}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(report,indent=2,allow_nan=False))
print("TUTORIAL_09_COMPLETE");print(json.dumps(report,indent=2,allow_nan=False))

## Exercises

1. Disable the causal mask and rerun the future-token check. Why can validation loss look deceptively good under teacher forcing?
2. Compare a repeated word at two positions before and after adding positional encoding.
3. Explain why source padding needs to be masked again in decoder cross-attention.
4. Compare teacher-forced token accuracy, known lexical-token accuracy, EOS rate, and exact-match rate. What does each miss?
5. Increase the training budget or vocabulary using only validation results. Reserve a new untouched test set if you already tuned after viewing this test output.

Outputs include five figures, held-out prediction CSV, a reload-checked checkpoint, and metrics. Larger mode uses 12,000/1,500/1,500 short pairs and 15 epochs; it still does not use the entire corpus. CPU quick mode is the default validation target; test GPU and larger mode separately.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_09/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

Adapted and reworked for independent Colab use from [PEESE Tutorial 09](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_09_Transformers_Implementation), Copyright (c) 2025 PEESE. The upstream MIT notice is retained in the repository `LICENSE`. This adaptation does not imply course endorsement.

References: [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [PyTorch MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html), [ManyThings bilingual pairs and attribution](https://www.manythings.org/anki/), and [Tatoeba](https://tatoeba.org/). Dataset terms are separate from the MIT code license; retain the downloaded archive's attribution. No dataset or trained weights are redistributed here.

This adaptation replaces the source GRU model with a Transformer, avoids incompatible torchtext downgrades, uses one consistent tokenizer/split, separates PAD/UNK/BOS/EOS, masks source padding, token-weights metrics, and verifies autoregressive inference plus checkpoint reload.